# Goal 7 — Simple data-drift check

Incoming batches can slowly stop looking like the training data. I use a **Kolmogorov–Smirnov (KS)** test per numeric feature (`scipy.stats.ks_2samp`). If enough features have a low p-value, I flag the batch as drifted.

I’ll check:
1. **Control** — holdout split from the same dataset (should look fine)
2. **Deliberate shift** — multiply/shift a few columns (should warn)


In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\saksh\OneDrive - University of Keele\Desktop\Ecommerce_project")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sklearn.model_selection import train_test_split

from src.config.settings import load_config
from src.data.loader import load_data
from src.evaluation.drift import check_dataframe_drift, check_drift
from src.features.engineering import add_engineered_features


In [ ]:
config = load_config()
df = add_engineered_features(load_data())
train_df, holdout_df = train_test_split(
    df,
    test_size=config["test_size"],
    random_state=config["random_seed"],
)

control = check_dataframe_drift(train_df, holdout_df)
print("CONTROL:", control["message"])
print(f"  flagged {control['n_drifted']}/{control['n_features_checked']} features")
control["details"].head(10)


In [ ]:
shifted = holdout_df.copy()
for col in ["PageValues", "BounceRates", "ExitRates", "ProductRelated"]:
    if col in shifted.columns:
        shifted[col] = shifted[col] * 5 + 10

print("Single-feature PageValues drifted?", check_drift(train_df["PageValues"], shifted["PageValues"]))

drifted = check_dataframe_drift(train_df, shifted)
print("SHIFTED:", drifted["message"])
print(f"  flagged {drifted['n_drifted']}/{drifted['n_features_checked']} features")
print("  drifted features:", drifted["drifted_features"])
drifted["details"].head(10)


## Takeaway

The checker stays quiet on a normal holdout split, and lights up when I deliberately shift several numeric columns. That’s enough for a student-project early warning: if a new batch looks drifted, dig into those features and consider retraining.

Helper: `src/evaluation/drift.py`  
Self-test: `python -m src.evaluation.run_drift_self_test`  
Dashboard: Home page shows a green/red drift status (control sample vs a simulated shifted sample).
